# 07 — Demo: dự đoán thử 1 ảnh X-quang bất kỳ

Không phải nhiệm vụ chính của đề tài (xem mục 1 README) — chỉ để minh hoạ trực quan model đang hoạt động ra sao trên 1 ảnh cụ thể, thay vì chỉ nhìn con số accuracy/macro-F1.

Tự động dùng **MedCLIP + LoRA đã train** nếu đã chạy `04_train_lora.ipynb` trước đó (đọc `outputs/checkpoints/lora_medclip.pt`); nếu chưa train checkpoint nào thì tự rơi về **MedCLIP zero-shot** (có in cảnh báo, không lỗi).

Yêu cầu: đã chạy `01_prepare_split.ipynb` trước đó.

In [ ]:
# Cell cài đặt — chạy trên Google Colab.
# Bỏ qua nếu chạy local (đã cài sẵn requirements + có sys.path đúng).

# 1) Mount Google Drive (nếu dataset/checkpoint lưu trên Drive)
# from google.colab import drive
# drive.mount('/content/drive')

# 2) Clone / trỏ tới thư mục project (sửa lại đường dẫn cho đúng chỗ bạn để code)
PROJECT_ROOT = '/content/drive/MyDrive/Code'  # <-- sửa lại nếu khác

# 3) Cài MedCLIP (editable) + dependency của src/
# !pip install -e {PROJECT_ROOT}/MedCLIP --no-deps -q
# !pip install -r {PROJECT_ROOT}/src/requirements.txt -q

# 4) Thêm src/ vào sys.path để import được các module (configs, data, models, pipelines...)
import sys
sys.path.insert(0, f'{PROJECT_ROOT}/src')


### Bước 1 — Load model (chạy 1 lần, dùng lại cho nhiều ảnh bên dưới)

In [ ]:
from predict import load_predictor

predictor = load_predictor()  # use_lora=True mặc định — tự rơi về zero-shot nếu chưa có checkpoint


### Bước 2 — Chọn ảnh để dự đoán

Cách 1: tải ảnh lên trực tiếp trên Colab (bỏ comment cell dưới). Cách 2 (mặc định): lấy ngẫu nhiên 1 ảnh từ `test.csv` để không cần chuẩn bị ảnh riêng.

In [ ]:
# Cách 1 — tải ảnh của riêng bạn lên (chỉ chạy được trên Colab)
# from google.colab import files
# uploaded = files.upload()
# image_path = list(uploaded.keys())[0]

# Cách 2 (mặc định) — lấy ngẫu nhiên 1 ảnh từ test set kèm nhãn thật để đối chiếu
import os
import pandas as pd
from configs import config as cfg

test_df = pd.read_csv(cfg.TEST_CSV)
row = test_df.sample(n=1).iloc[0]
image_path = os.path.join(cfg.IMAGES_DIR, row['image_filename'])
print('Ảnh:', row['image_filename'], '— nhãn thật:', cfg.CLASS_NAMES[row['label']])


### Bước 3 — Dự đoán + hiển thị

In [ ]:
from predict import predict_image
from PIL import Image
import matplotlib.pyplot as plt

result = predict_image(image_path, predictor)

plt.figure(figsize=(4, 4))
plt.imshow(Image.open(image_path), cmap='gray')
plt.axis('off')
plt.title(f"Dự đoán: {result['pred_class']}")
plt.show()

result['probs']
